# Session 3: Character Embeddings

Session 2 ended with: "What if instead of a V×V matrix, we represented each character as a small vector?"

This is that.

1. Recap: load data, build the counting bigram (our ground truth)
2. Full-rank model: V×V matrix (d=V) — same as Session 2
3. Factored model: V×d embedding + d×V output (d < V) — the new idea
4. Compare probabilities: does the compressed model recover the same distribution?
5. Visualize: plot the 2D character embeddings — what did the model learn?
6. The punchline: same trick works for words, and that's how real LLMs start

In [ ]:
import random
import math
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from tqdm import trange

random.seed(42)
torch.manual_seed(42)

## 1. Load Data (same as Session 2)

In [ ]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
response = urllib.request.urlopen(url)
names = [n.strip().lower() for n in response.read().decode('utf-8').strip().split('\n') if n.strip()]

chars = sorted(set("".join(names)))
stoi = {'.': 0}
for i, c in enumerate(chars):
    stoi[c] = i + 1
itos = {i: c for c, i in stoi.items()}
V = len(stoi)

# Training pairs
xs, ys = [], []
for name in names:
    chs = ['.'] + list(name) + ['.']
    for i in range(len(chs) - 1):
        xs.append(stoi[chs[i]])
        ys.append(stoi[chs[i + 1]])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

print(f"Names: {len(names):,}")
print(f"Vocab: {V} characters")
print(f"Training bigrams: {len(xs):,}")

In [ ]:
# Counting bigram (ground truth)
counts = [[0] * V for _ in range(V)]
for name in names:
    chs = ['.'] + list(name) + ['.']
    for i in range(len(chs) - 1):
        counts[stoi[chs[i]]][stoi[chs[i + 1]]] += 1

P_counts = torch.zeros(V, V)
for i in range(V):
    row_total = sum(counts[i])
    if row_total > 0:
        for j in range(V):
            P_counts[i][j] = counts[i][j] / row_total

# Counting model loss
counting_loss = F.cross_entropy(
    # treat P_counts rows as logits (log of probs)
    P_counts.log()[xs], ys
).item()
print(f"Counting bigram loss: {counting_loss:.4f}")

## 2. Full-Rank Model: V×V (Session 2 recap)

This is what we did in Session 2: one big matrix W of shape V×V.  
Row i gives the logits for "what comes after character i".  
Parameters: 27×27 = 729

In [ ]:
torch.manual_seed(42)
W_full = torch.randn((V, V), requires_grad=True)

lr = 50.0
for step in trange(300, desc="Full V×V"):
    logits = W_full[xs]
    loss = F.cross_entropy(logits, ys)
    W_full.grad = None
    loss.backward()
    with torch.no_grad():
        W_full -= lr * W_full.grad

print(f"Final loss: {loss.item():.4f}")

# Convert logits to probabilities
P_full = W_full.detach().softmax(dim=1)

## 3. Factored Model: Embeddings (THE NEW IDEA)

Instead of one V×V matrix, we split it into two smaller matrices:

```
C : V × d   (the "embedding" — each character becomes a d-dim vector)
W : d × V   (the "output"   — turns the vector back into scores)
```

To get logits for character i:
1. Look up row i of C → a vector of d numbers
2. Multiply by W → a vector of V scores (logits)

This is: `logits = C @ W` (a V×V matrix, but built from two small ones)

- If **d = V**, this can represent any V×V matrix.
- If **d < V**, the model is forced to compress: similar characters must end up with similar embedding vectors.

Key insight: we build the FULL logit matrix `C @ W`, then index into it. Autograd sees both C and W as leaf tensors and can compute gradients for both.

In [ ]:
def train_factored(d, steps=2000, lr=10.0, seed=42):
    """Train a factored bigram: C (V×d) @ W (d×V)."""
    torch.manual_seed(seed)
    # Small init — two matrices multiplied together means gradients
    # flow through a product, so we keep values small to start
    C = (torch.randn((V, d)) * 0.01).requires_grad_(True)
    W = (torch.randn((d, V)) * 0.01).requires_grad_(True)

    n_params = V * d + d * V
    print(f"d = {d:>2d}  →  params: {V}×{d} + {d}×{V} = {n_params}  (vs {V * V} for full V×V)")

    losses = []
    for step in trange(steps, desc=f"d={d}", leave=False):
        # Build the full V×V logit matrix, then index into it
        logit_matrix = C @ W              # (V, V)
        logits = logit_matrix[xs]         # (N, V) — look up row per input
        loss = F.cross_entropy(logits, ys)

        C.grad = None
        W.grad = None
        loss.backward()
        with torch.no_grad():
            C -= lr * C.grad
            W -= lr * W.grad

        losses.append(loss.item())

    print(f"  final loss = {losses[-1]:.4f}")
    P = (C @ W).detach().softmax(dim=1)
    return C.detach(), W.detach(), P, losses

In [ ]:
# Train at several dimensions to show the compression tradeoff
results = {}
for d in [2, 5, 10, 15, 27]:
    C, W_out, P, losses = train_factored(d)
    results[d] = {'C': C, 'W': W_out, 'P': P, 'losses': losses}
    print()

## 4. Compare: Do They Recover the Same Distribution?

In [ ]:
print(f"{'model':>20s}  {'params':>8s}  {'loss':>8s}  {'max diff':>10s}  {'mean diff':>10s}")
print(f"{'-' * 62}")

# Counting baseline
print(f"{'counting':>20s}  {'—':>8s}  {counting_loss:>8.4f}  {'—':>10s}  {'—':>10s}")

# Full V×V
diff = (P_counts - P_full).abs()
full_loss = F.cross_entropy(W_full.detach()[xs], ys).item()
print(f"{'full V×V (d=27)':>20s}  {V * V:>8d}  {full_loss:>8.4f}"
      f"  {diff.max().item():>10.6f}  {diff.mean().item():>10.6f}")

# Factored models
for d in [2, 5, 10, 15, 27]:
    P = results[d]['P']
    diff = (P_counts - P).abs()
    final_loss = results[d]['losses'][-1]
    params = 2 * V * d
    print(f"{'factored d=' + str(d):>20s}  {params:>8d}  {final_loss:>8.4f}"
          f"  {diff.max().item():>10.6f}  {diff.mean().item():>10.6f}")

At d=27, the factored model has the same capacity as the V×V matrix and recovers the exact same probabilities. As d shrinks, the model is forced to compress — but even d=10 gets very close.

d=2 is too small to represent all the patterns, but it's perfect for visualization: each character IS a 2D point we can plot.

## 5. Probability Comparison: d=2 vs Counting

Let's look at specific rows of the probability matrix to see where d=2 succeeds and where it struggles.

In [ ]:
P_d2 = results[2]['P']

def compare_row(char, P_true, P_model, stoi, itos, top_k=10, model_name='d=2'):
    """Print side-by-side probability comparison for a given context character."""
    idx = stoi[char]
    c_row = P_true[idx]
    p_row = P_model[idx]
    print(f"\nP(next | '{char}') — top {top_k}:")
    print(f"  {'char':>6s}  {'counting':>10s}  {model_name:>10s}  {'diff':>10s}")
    print(f"  {'-' * 42}")
    top_idx = c_row.argsort(descending=True)[:top_k]
    for i in top_idx:
        ch = itos[i.item()]
        pc = c_row[i].item()
        pp = p_row[i].item()
        print(f"  {repr(ch):>6s}  {pc:>10.4f}  {pp:>10.4f}  {abs(pc - pp):>10.6f}")

# What starts a name?
compare_row('.', P_counts, P_d2, stoi, itos)

# What follows 'm'?
compare_row('m', P_counts, P_d2, stoi, itos)

# 'q' is interesting — almost always followed by 'u'
compare_row('q', P_counts, P_d2, stoi, itos)

## 6. Visualize Character Embeddings (d=2)

With d=2, each character is literally a point in 2D space. Characters that appear in similar contexts (similar "what comes next" distributions) should end up nearby.

In [ ]:
C_2d = results[2]['C'].numpy()

# Define character groups for coloring
vowels = set('aeiou')
common_consonants = set('tnrsld')
rare_consonants = set('qxz')

def get_group(ch):
    if ch == '.':
        return 'start/end (.)'
    elif ch in vowels:
        return 'vowels'
    elif ch in rare_consonants:
        return 'rare (q,x,z)'
    elif ch in common_consonants:
        return 'common consonants'
    else:
        return 'other consonants'

group_colors = {
    'start/end (.)': '#e74c3c',
    'vowels': '#3498db',
    'common consonants': '#2ecc71',
    'other consonants': '#95a5a6',
    'rare (q,x,z)': '#e67e22',
}

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

for i in range(V):
    ch = itos[i]
    group = get_group(ch)
    color = group_colors[group]
    ax.scatter(C_2d[i, 0], C_2d[i, 1], c=color, s=200, zorder=3,
              edgecolors='white', linewidth=0.5)
    ax.annotate(ch, (C_2d[i, 0], C_2d[i, 1]),
                fontsize=12, fontweight='bold', ha='center', va='center',
                color='white', zorder=4)

# Legend
for group, color in group_colors.items():
    ax.scatter([], [], c=color, s=80, label=group)
ax.legend(loc='best', fontsize=9, framealpha=0.9)

ax.set_xlabel("Embedding dimension 1", fontsize=11)
ax.set_ylabel("Embedding dimension 2", fontsize=11)
ax.set_title("Character Embeddings (d=2) — Learned from Bigram Data\n"
             "Characters that appear in similar contexts end up nearby",
             fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("char_embeddings_2d.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Loss Curves by Embedding Dimension

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)

colors = {2: '#e74c3c', 5: '#e67e22', 10: '#3498db', 15: '#2ecc71', 27: '#9b59b6'}
for d in [2, 5, 10, 15, 27]:
    losses = results[d]['losses']
    ax.plot(losses, label=f'd={d} ({2 * V * d} params)', color=colors[d], linewidth=1.5)

ax.axhline(y=counting_loss, color='black', linestyle='--', linewidth=1,
           label=f'counting ({counting_loss:.4f})')
ax.set_xlabel("Training step", fontsize=11)
ax.set_ylabel("Loss (cross-entropy)", fontsize=11)
ax.set_title("Embedding Dimension vs Loss\n"
             "Higher d = more capacity = closer to counting baseline", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(bottom=2.0)
plt.tight_layout()
plt.savefig("loss_by_dimension.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. The Punchline

In [ ]:
print(f"""
What we just did:
  - Took the V×V bigram matrix from Session 2
  - Factored it into C (V×d) and W (d×V)
  - Each character is now a vector of d numbers — an EMBEDDING
  - Characters that behave similarly get similar vectors
  - Even d=2 captures most of the structure (loss {results[2]['losses'][-1]:.4f} vs {counting_loss:.4f})

The parameter savings:
  V×V full matrix:  {V}×{V} = {V * V:>6d} parameters
  Factored d=10:    {V}×10 + 10×{V} = {2 * V * 10:>6d} parameters  ({2 * V * 10 / (V * V) * 100:.0f}% of full)
  Factored d=2:     {V}×2  + 2×{V}  = {2 * V * 2:>6d} parameters  ({2 * V * 2 / (V * V) * 100:.0f}% of full)

Now scale this up to words:
  V = 50,000 words (GPT-2 vocab)
  Full V×V:     50000² = {50000 ** 2:,} parameters (10 GB)
  Factored d=768: 50000×768×2 = {50000 * 768 * 2:,} parameters ({50000 * 768 * 2 * 4 / 1e6:.0f} MB)

That's the same trick. Every language model starts with an embedding table.
The embedding table in GPT-2 IS this C matrix, just bigger.

Next: what if we look at MORE than one previous character?
Concatenate multiple embeddings → add a hidden layer → that's Bengio (2003).
""")